# Tool Use 기초 (3) — 멀티턴 대화 루프

**Skilljar Lessons 07-08 대응**

이 노트북에서 다루는 내용:
1. 헬퍼 함수 리팩토링 (`add_user_message`, `add_assistant_message`)
2. `chat()` 함수에 tools 파라미터 추가
3. `text_from_message()` 헬퍼
4. `run_tool()` — 도구 라우팅 함수
5. `run_tools()` — 복수 도구 블록 처리
6. `run_conversation()` — 자동 멀티턴 루프
7. 단일 도구 테스트
8. 복수 도구 테스트 (2개 도구 연쇄)

In [ ]:
# ── Setup ──────────────────────────────────────────────
import anthropic
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

## §1. 헬퍼 함수 리팩토링 (Refactored Helpers)

멀티턴 대화를 위해 메시지 관리 함수를 정리합니다.  
`add_assistant_message`는 API의 `Message` 객체도 직접 받을 수 있습니다.

In [ ]:
def add_user_message(messages, content):
    """사용자 메시지를 대화 히스토리에 추가합니다."""
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, response):
    """
    어시스턴트 메시지를 대화 히스토리에 추가합니다.
    response가 Message 객체이면 .content를 사용합니다.
    """
    if isinstance(response, anthropic.types.Message):
        messages.append({"role": "assistant", "content": response.content})
    else:
        messages.append({"role": "assistant", "content": response})

## §2. chat() 함수 업데이트 (Updated chat Function)

`tools` 파라미터를 추가하여 도구 사용이 가능한 API 호출을 합니다.

In [ ]:
def chat(messages, tools=None, system=None):
    """
    Claude API를 호출합니다.

    Args:
        messages: 대화 히스토리
        tools: 도구 스키마 리스트 (optional)
        system: 시스템 프롬프트 (optional)

    Returns:
        API Message 응답
    """
    kwargs = {
        "model": MODEL,
        "max_tokens": 4096,
        "messages": messages,
    }
    if tools:
        kwargs["tools"] = tools
    if system:
        kwargs["system"] = system

    return client.messages.create(**kwargs)

## §3. text_from_message() 헬퍼 (Text Extraction Helper)

응답에서 텍스트 블록만 추출하는 편의 함수입니다.

In [ ]:
def text_from_message(response):
    """Message 응답에서 텍스트 블록들을 합쳐 반환합니다."""
    texts = []
    for block in response.content:
        if block.type == "text":
            texts.append(block.text)
    return "\n".join(texts)

## 도구 함수 및 스키마 정의 (Tool Functions & Schemas)

이 노트북에서는 **2개의 도구**를 사용합니다:
1. `get_current_datetime` — 현재 날짜/시간 조회
2. `add_duration_to_datetime` — 날짜에 기간 더하기

In [ ]:
# ── Tool Functions ──

def get_current_datetime(date_format: str = "%Y-%m-%d %H:%M:%S") -> str:
    valid_formats = [
        "%Y-%m-%d %H:%M:%S", "%Y-%m-%d", "%H:%M:%S",
        "%H:%M", "%Y/%m/%d", "%m/%d/%Y",
    ]
    if date_format not in valid_formats:
        raise ValueError(
            f"Invalid date format: {date_format}. Valid formats: {valid_formats}"
        )
    return datetime.now().strftime(date_format)


def add_duration_to_datetime(
    date_str: str,
    duration_days: int = 0,
    duration_hours: int = 0,
    duration_minutes: int = 0,
) -> str:
    """
    주어진 날짜/시간에 기간을 더하여 새로운 날짜/시간을 반환합니다.

    Args:
        date_str: "%Y-%m-%d %H:%M:%S" 형식의 날짜 문자열
        duration_days: 더할 일수
        duration_hours: 더할 시간수
        duration_minutes: 더할 분수

    Returns:
        계산된 날짜/시간 문자열
    """
    if not date_str:
        raise ValueError("date_str cannot be empty")
    dt = datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")
    dt += timedelta(
        days=duration_days,
        hours=duration_hours,
        minutes=duration_minutes,
    )
    return dt.strftime("%Y-%m-%d %H:%M:%S")


# 테스트
print("현재:", get_current_datetime())
print("+10일:", add_duration_to_datetime(get_current_datetime(), duration_days=10))

In [ ]:
# ── Tool Schemas ──

get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": (
        "Returns the current date and time in a specified format. "
        "Defaults to '%Y-%m-%d %H:%M:%S'."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": (
                    "The format string for the date/time output. "
                    "Supported: '%Y-%m-%d %H:%M:%S', '%Y-%m-%d', '%H:%M:%S', "
                    "'%H:%M', '%Y/%m/%d', '%m/%d/%Y'."
                ),
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
}

add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": (
        "Adds a duration (days, hours, minutes) to a given datetime string. "
        "Input datetime must be in '%Y-%m-%d %H:%M:%S' format."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "date_str": {
                "type": "string",
                "description": "The starting datetime in '%Y-%m-%d %H:%M:%S' format.",
            },
            "duration_days": {
                "type": "integer",
                "description": "Number of days to add. Defaults to 0.",
                "default": 0,
            },
            "duration_hours": {
                "type": "integer",
                "description": "Number of hours to add. Defaults to 0.",
                "default": 0,
            },
            "duration_minutes": {
                "type": "integer",
                "description": "Number of minutes to add. Defaults to 0.",
                "default": 0,
            },
        },
        "required": ["date_str"],
    },
}

tools = [get_current_datetime_schema, add_duration_to_datetime_schema]

## §4. run_tool() — 도구 라우팅 함수 (Tool Routing)

도구 이름으로 실제 Python 함수를 찾아 실행하는 라우터입니다.

In [ ]:
def run_tool(tool_name, tool_input):
    """
    도구 이름에 해당하는 함수를 실행합니다.

    Args:
        tool_name: 도구 이름 (str)
        tool_input: 도구 입력 인자 (dict)

    Returns:
        도구 실행 결과 (str)
    """
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    else:
        return f"Error: Unknown tool '{tool_name}'"

## §5. run_tools() — 복수 도구 블록 처리 (Processing Tool Blocks)

응답에 여러 `tool_use` 블록이 있을 수 있으므로, 모두 순회하면서  
실행 → `tool_result` 블록 생성을 반복합니다.

In [ ]:
def run_tools(response):
    """
    응답의 모든 tool_use 블록을 실행하고 tool_result 리스트를 반환합니다.

    Args:
        response: API Message 응답

    Returns:
        tool_result 블록 리스트
    """
    tool_results = []

    for block in response.content:
        if block.type != "tool_use":
            continue

        tool_name = block.name
        tool_input = block.input

        print(f"  🔧 Running tool: {tool_name}({tool_input})")

        try:
            result = run_tool(tool_name, tool_input)
            print(f"  ✅ Result: {result}")
            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(result),
                }
            )
        except Exception as e:
            error_msg = f"Error executing {tool_name}: {str(e)}"
            print(f"  ❌ {error_msg}")
            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": error_msg,
                    "is_error": True,
                }
            )

    return tool_results

## §6. run_conversation() — 자동 멀티턴 루프 (Conversation Loop)

핵심 루프 로직:
```
while True:
    1. Claude에게 메시지 전송
    2. stop_reason이 "tool_use"가 아니면 → break (최종 응답)
    3. 도구 실행 → tool_result를 messages에 추가
    4. 다시 1번으로
```

In [ ]:
def run_conversation(user_message, tools, system=None, verbose=True):
    """
    사용자 메시지로 시작하여 도구 사용이 완료될 때까지
    자동으로 멀티턴 대화를 실행합니다.

    Args:
        user_message: 사용자 질문/요청 (str)
        tools: 도구 스키마 리스트
        system: 시스템 프롬프트 (optional)
        verbose: 상세 출력 여부

    Returns:
        (final_response, messages) 튜플
    """
    messages = []
    add_user_message(messages, user_message)

    if verbose:
        print(f"User: {user_message}")
        print("=" * 60)

    turn = 0
    while True:
        turn += 1
        if verbose:
            print(f"\n--- Turn {turn} ---")

        response = chat(messages, tools=tools, system=system)

        if verbose:
            print(f"stop_reason: {response.stop_reason}")

        # 도구 사용이 아니면 → 최종 응답
        if response.stop_reason != "tool_use":
            if verbose:
                print(f"\nClaude: {text_from_message(response)}")
            return response, messages

        # assistant 응답을 대화에 추가
        add_assistant_message(messages, response)

        # 도구 실행
        tool_results = run_tools(response)

        # tool_result를 대화에 추가
        messages.append({"role": "user", "content": tool_results})

    return response, messages

## §7. 테스트: 단일 도구 (Single Tool Test)

"What is the exact time right now?" → `get_current_datetime` 1회 호출

In [ ]:
response, messages = run_conversation(
    "What is the exact time right now?",
    tools=tools,
)

## §8. 테스트: 복수 도구 연쇄 (Multi-Tool Chain Test)

"What day is 103 days from today?"  
→ Turn 1: `get_current_datetime` (오늘 날짜 확인)  
→ Turn 2: `add_duration_to_datetime` (103일 더하기)  
→ Turn 3: 최종 답변

In [ ]:
response, messages = run_conversation(
    "What day is 103 days from today?",
    tools=tools,
)